In [1]:
# Instale as dependências deste notebook no kernel atual, se necessário.
%pip install requests pandas


   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------  9.7/9.8 MB 49.9 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 45.5 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---------------------------------------- 12.7/12.7 MB 65.8 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [tzdata]
   ----- ---------------------------------- 1/8 [tzdata]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Exploração inicial — API da Câmara

A API fornece JSON/XML e as listagens paginam resultados. Faça chamadas pequenas; não remova filtros nem percorra o histórico todo nesta etapa. Consulte `docs/fontes_e_endpoints.md`.

In [2]:
from datetime import datetime, timezone
import requests
import pandas as pd
from pprint import pprint

BASE = 'https://dadosabertos.camara.leg.br/api/v2'
session = requests.Session()
session.headers.update({'Accept': 'application/json', 'User-Agent': 'exploracao-painel-legislativo/0.1'})

def get_json(path, params=None):
    response = session.get(f'{BASE}/{path.lstrip("/")}', params=params, timeout=30)
    print(response.status_code, response.url)
    response.raise_for_status()
    return response.json()


## 1. Inventário das rotas OpenAPI

A especificação é a fonte mais confiável para listar caminhos e operações. Filtre por tags para priorizar domínios.

In [3]:
spec = get_json('api-docs')
print(spec['info'])
routes = []
for path, operations in spec['paths'].items():
    for method, operation in operations.items():
        if method.lower() in {'get', 'post', 'put', 'delete'}:
            routes.append({'tag': ', '.join(operation.get('tags', [])), 'method': method.upper(), 'path': path, 'summary': operation.get('summary', '')})
routes_df = pd.DataFrame(routes).sort_values(['tag', 'path']).reset_index(drop=True)
display(routes_df)


200 https://dadosabertos.camara.leg.br/api/v2/api-docs
{'description': '**Versão: 0.4.344 - 08/18/2026 02:19 PM** - _confira o que mudou na [página de Notícias](http://dadosabertos.camara.leg.br/news/noticias.html)_\n\nBem vindo à nova versão do serviço **Dados Abertos** da Câmara dos Deputados!\n\nEsta versão pode entregar dados puros em formatos JSON e XML, e tenta se aderir ao máximo à arquitetura REST.\n\nNesta página você pode conhecer e experimentar as URLs de acesso aos dados, os parâmetros de *query string* que podem ser aplicados para filtrar e selecionar resultados, e as estruturas de dados que são retornadas.\n\nAlém do método HTTP GET, você também pode usar o método HEAD com qualquer dos serviços.\n\nPor padrão, todos os serviços de listagens retornam 15 itens, e o limite por requisição é de 100 itens.\n\n **ATENÇÃO:** Esta versão é ainda incompleta, sujeita a mudanças e não substitui a [versão original do **Dados Abertos**](http://www2.camara.leg.br/transparencia/dados-abe

,tag,method,path,summary
0,Blocos,GET,/blocos,Lista de dados sobre os blocos partidários
1,Blocos,GET,/blocos/{id},Informações sobre um bloco partidário específico
2,Blocos,GET,/blocos/{id}/partidos,Lista de partidos do bloco.
3,Deputados,GET,/deputados,"Listagem e busca de deputados, segundo critérios"
4,Deputados,GET,/deputados/{id},Informações detalhadas sobre um deputado espec...
...,...,...,...,...
74,Órgãos,GET,/orgaos/{id}/eventos,Os eventos ocorridos ou previstos em um órgão ...
75,Órgãos,GET,/orgaos/{id}/membros,Lista de cargos de um órgão e parlamentares qu...
76,Órgãos,GET,/orgaos/{id}/votacoes,Informações detalhadas sobre votações de um ór...
77,"Órgãos, Referências",GET,/referencias/orgaos/codSituacao,As situações em que órgãos podem se encontrar


## 2. Amostras pequenas: deputados, proposições e votações

Registre chaves, campos aninhados e paginação. Compare o registro resumido com o detalhe individual.

In [4]:
deputies = get_json('deputados', {'itens': 5, 'pagina': 1})
print('campos envelope:', deputies.keys(), '| links:', deputies.get('links'))
display(pd.json_normalize(deputies.get('dados', [])))

deputy = (deputies.get('dados') or [None])[0]
if deputy:
    deputy_detail = get_json(f"deputados/{deputy['id']}")
    pprint(deputy_detail)

for path in ['proposicoes', 'votacoes']:
    sample = get_json(path, {'itens': 3, 'pagina': 1})
    print('\n', path, 'links:', sample.get('links'))
    display(pd.json_normalize(sample.get('dados', [])))


200 https://dadosabertos.camara.leg.br/api/v2/deputados?itens=5&pagina=1
campos envelope: dict_keys(['dados', 'links']) | links: [{'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?itens=5&pagina=1'}, {'rel': 'next', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=2&itens=5'}, {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=5'}, {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=103&itens=5'}]


,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.acaciofavacho@camara.leg.br
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,Adail Filho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AM,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adailfilho@camara.leg.br
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adilsonbarroso@camara.leg.br
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/part...,BA,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adolfoviana@camara.leg.br
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adrianaventura@camara.leg.br


200 https://dadosabertos.camara.leg.br/api/v2/deputados/204379
{'dados': {'cpf': '74287028287',
           'dataFalecimento': None,
           'dataNascimento': '1983-09-28',
           'escolaridade': 'Superior',
           'id': 204379,
           'municipioNascimento': 'Macapá',
           'nomeCivil': 'ACÁCIO DA SILVA FAVACHO NETO',
           'redeSocial': ['https://twitter.com/acaciofavacho',
                          'https://www.facebook.com/deputadofederalacaciofavacho',
                          'https://youtube.com/channel/UCWNbptIjAriL2pt8H3W6CGw'],
           'sexo': 'M',
           'ufNascimento': 'AP',
           'ultimoStatus': {'condicaoEleitoral': 'Titular',
                            'data': '2023-02-01',
                            'descricaoStatus': None,
                            'email': None,
                            'gabinete': {'andar': '4',
                                         'email': 'dep.acaciofavacho@camara.leg.br',
                             

,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,117992,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1156,2003,Estabelece limite para a cobrança de juros no ...,2003-06-02T19:05
1,125816,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1552,2003,"Altera a Lei nº 8.036, de 11 de maio de 1990,...",2003-07-24T18:22
2,129503,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1779,2003,"Acrescenta dispositivo à Lei nº 10.406, de 10 ...",2003-08-21T18:57


200 https://dadosabertos.camara.leg.br/api/v2/votacoes?itens=3&pagina=1

 votacoes links: [{'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/votacoes?itens=3&pagina=1'}, {'rel': 'next', 'href': 'https://dadosabertos.camara.leg.br/api/v2/votacoes?pagina=2&itens=3'}, {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/votacoes?pagina=1&itens=3'}, {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/votacoes?pagina=371&itens=3'}]


,id,uri,data,dataHoraRegistro,siglaOrgao,uriOrgao,uriEvento,proposicaoObjeto,uriProposicaoObjeto,descricao,aprovacao
0,2611313-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-09-03,2026-09-03T17:29:39,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/even...,None,None,Aprovada a Redação Final assinada pelo relator...,1
1,2611313-31,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-09-03,2026-09-03T17:28:34,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/even...,None,None,Aprovada a Subemenda Substitutiva ao Projeto d...,1
2,2634392-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2026-09-03,2026-09-03T16:49:29,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/even...,None,None,Aprovada a Redação Final.,1


## 3. Relacionamentos e despesas

A documentação alerta que despesas sem filtro temporal retornam somente seis meses recentes. A amostra abaixo mantém essa chamada pequena e testa uma página. Depois identifique na especificação parâmetros de histórico e limites de página.

In [5]:
if deputy:
    expenses = get_json(f"deputados/{deputy['id']}/despesas", {'itens': 5, 'pagina': 1})
    print('links:', expenses.get('links'))
    display(pd.json_normalize(expenses.get('dados', [])))

# Para um endpoint selecionado, inspecione parâmetros e schema de resposta:
for path in ['/deputados', '/proposicoes', '/votacoes']:
    operation = spec['paths'].get(path, {}).get('get', {})
    print('\n', path, operation.get('summary'))
    pprint(operation.get('parameters', []))


200 https://dadosabertos.camara.leg.br/api/v2/deputados/204379/despesas?itens=5&pagina=1
links: [{'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados/204379/despesas?itens=5&pagina=1'}]


""



 /deputados Listagem e busca de deputados, segundo critérios
[{'description': 'Número(s) identificador(es) de um deputado, separados por '
                 'vírgulas.',
  'explode': False,
  'in': 'query',
  'name': 'id',
  'required': False,
  'schema': {'type': 'array'},
  'style': 'form'},
 {'description': 'Parte nome parlamentar.',
  'in': 'query',
  'name': 'nome',
  'required': False,
  'schema': {'type': 'string'}},
 {'description': 'Número(s) identificador(es) de uma ou mais legislatura(s) de '
                 'que os parlamentares tenham participado, separados por '
                 'vírgulas.',
  'explode': False,
  'in': 'query',
  'name': 'idLegislatura',
  'required': False,
  'schema': {'type': 'array'},
  'style': 'form'},
 {'description': 'Uma ou mais sigla(s) de unidades federativas (estados e '
                 'Distrito Federal). Uma lista de siglas válidas pode ser '
                 'obtida em `/referencias/deputados/siglaUf`. Se ausente, '
                 'serã

## Anotações de descoberta

Preencha após executar: quantidade/campos, campos nulos, paginação, erros, relações confirmadas por ID e possíveis diferenças entre resposta resumida/detalhada. Guarde exemplos anonimizados/pequenos aqui; a carga bruta pertence à etapa Bronze futura.

## Anotações dos outputs — 2026-10-09

- A especificação retornou HTTP 200 e descreveu 79 operações GET. A Câmara usa envelope `dados`/`links`; com `itens=3`, os links `last` indicaram 103 páginas para deputados, 4.897 para proposições e 371 para votações (contagens do snapshot, não contrato estável).
- IDs de deputado e proposição aparecem no caminho de detalhe; tratamos `id` como chave candidata forte. IDs de votação são texto, por exemplo `2611313-34`.
- O detalhe da proposição `117992`, seus autores e suas tramitações responderam HTTP 200; foram observados 1 autor e 17 tramitações. O endpoint de votações dessa proposição retornou zero itens.
- A votação `2611313-34` respondeu ao detalhe, mas sua lista de votos também estava vazia. Estas amostras não indicam ausência geral de votos nominais.
- Despesas retornaram zero itens tanto sem filtro como com `ano=2026&mes=8` para quatro deputados. Cobertura/atualização do período segue por confirmar; comparar outros períodos com os arquivos anuais de cotas.
- Ver o mapa de chaves em `docs/chaves_relacionamentos.md` e o catálogo completo em `docs/catalogo_recursos.csv`.
